# AfriVoices — v8 CIBLE (tranche 2, langues faibles, depuis le DRIVE)

Depart **v7-soup** (fusion v5-t3+v6-t1, meilleur modele unique : dev+LM 0.3469).
Objectif : specialiser sur kln/mas/som (70% du mix) sans oublier les fortes (30%),
sur la tranche 2 (donnees fraiches), lues **depuis le Drive** (`afrivoices/anvke/`)
— plus de telechargement HF, resilient aux coupures.

**Prerequis Drive** : `MyDrive/afrivoices/anvke/` contient 5 raccourcis (ou dossiers)
kikuyu / Dholuo / Kalenjin / Maasai / Somali.

**Ordre** : sec.2 (config) -> sec.4 (Drive) -> sec.4.2 (swahili) -> sec.5 (Drive)
-> 5.1 -> 6 -> 7 -> 8. Garde-fou 8 : kln/mas/som en baisse nette, swa/kik/luo stables.

⚠️ Ne jamais mettre de token HF en clair. Panneau Secrets de Colab si besoin.

## 1. Dépendances

In [ ]:
!pip install -q "transformers>=4.44,<5" "datasets>=3.5,<4" "accelerate>=0.33" \
    "jiwer>=3.0" torchaudio soundfile ffmpeg-python librosa huggingface_hub hf_transfer
print("ok")

## 2. Config — REGLE `TRANCHE` ICI (0,1,2,... une par session)

In [ ]:
import os, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
os.environ["HF_HOME"]="/content/hf_cache"
os.environ["HF_DATASETS_CACHE"]="/content/hf_cache/datasets"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"     # nuisible sur connexion instable

from google.colab import drive
drive.mount("/content/drive")
import numpy as np, torch
# from huggingface_hub import login; login()   # PAS besoin : on lit le train depuis le Drive.
# (le dev officiel §4.1 vient du Drive aussi maintenant. Si un jour tu retournes sur hf://,
#  utilise le panneau Secrets de Colab, JAMAIS un token en clair.)

# ========== LE SEUL REGLAGE A CHANGER ENTRE LES SESSIONS ==========
TRANCHE = 2            # 0,1,2,... — une tranche par session
NB_TRANCHES = 6
# ===================================================================

BASE="/content/drive/MyDrive/afrivoices"
START_DIR = f"{BASE}/baobab-asr-v7-soup"      # depart = la soupe (meilleur modele unique)
OUT_DIR   = f"{BASE}/baobab-asr-v8-cible"     # sortie dediee

SAMPLING_RATE=16_000; MAX_AUDIO_S=20; SEED=142+TRANCHE; TEMPERATURE=1.6
os.makedirs(OUT_DIR, exist_ok=True); torch.manual_seed(SEED); np.random.seed(SEED)

LANG_HOURS={"swa":2979,"som":1002,"kik":754,"luo":723,"kln":521,"mas":505}
LANGS=list(LANG_HOURS.keys())
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print(f"GPU {p.name} {p.total_memory/1e9:.0f}Go")
else: print("pas de GPU !")
t,u,f=shutil.disk_usage("/content"); print(f"Disque libre : {f/1e9:.0f} Go")

# PROBS CIBLEES : langues faibles sur-echantillonnees (70%), fortes en anti-oubli (30%)
probs = {"kln": 0.28, "mas": 0.24, "som": 0.18, "luo": 0.12, "kik": 0.10, "swa": 0.08}
print("probs CIBLEES (faibles 70%):", probs)
assert os.path.isdir(START_DIR), f"modele de depart absent : {START_DIR} (v7-soup pas sauvegarde ?)"

## 3. Processor (vocab figé)

## 4. Repartition Anv-ke : train par tranches + eval = dev officiel

Les depots Anv-ke ont train/ et dev/ par type. L'entrainement tire du train/
(parquets i % NB_TRANCHES == TRANCHE) ; l'eval vient du **dev officiel** (fixe, toutes
tranches). La 4.0 verifie l'absence de fuite du test.

In [ ]:
# (auth non requise : lecture du train + dev depuis le Drive)

In [ ]:
# (hf_transfer desactive en section 2 ; rien a faire ici)

In [ ]:
import os, glob
ANVKE = "/content/drive/MyDrive/afrivoices/anvke"
assert os.path.isdir(ANVKE), f"introuvable: {ANVKE} (cree le dossier + 5 raccourcis dedans)"
print("dossiers vus dans anvke:", os.listdir(ANVKE))

ALIAS = {"kik":["kik","kikuyu","gikuyu"], "luo":["luo","dholuo"], "kln":["kln","kalenjin"],
         "mas":["mas","maasai"], "som":["som","somali"]}
lang_dirs = {}
for d in os.listdir(ANVKE):
    dl = d.lower()
    for lang, als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs:
            lang_dirs[lang] = os.path.join(ANVKE, d)
print("mapping langue -> dossier:", lang_dirs)
assert len(lang_dirs) == 5, f"seulement {len(lang_dirs)}/5 dossiers reconnus — verifie les noms"

KE_LANGS = list(lang_dirs.keys())
train_files, dev_files = {}, {}
for lang, root in lang_dirs.items():
    allp = sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    tr = [p for p in allp if "/train/" in p or os.path.basename(p).startswith("train")]
    dv = [p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
    if not tr and not dv:        # structure a plat -> tout en train
        tr = allp
    train_files[lang] = [p for i, p in enumerate(tr) if i % NB_TRANCHES == TRANCHE]
    dev_files[lang] = dv
    print(f"{lang}: {len(tr)} train ({len(train_files[lang])} tranche {TRANCHE}) | {len(dv)} dev")

### 4.0 — Filet anti-fuite : aucun clip du TEST dans ces donnees ?

Croise les filename des nouveaux parquets (echantillon) avec les ids du test. Un
chevauchement > 0 serait grave (entrainement sur le test = disqualifiant) — a signaler
aux organisateurs et a exclure rigoureusement.

In [6]:
import pandas as pd, glob as g, os
test_ids=set()
for pat in ["/content/drive/MyDrive/afrivoices/test/**/*.parquet",
            "/content/test_data/**/*.parquet"]:
    for pq in g.glob(pat, recursive=True):
        try: test_ids.update(pd.read_parquet(pq, columns=["id"])["id"].astype(str))
        except Exception: pass
print("ids test charges:", len(test_ids))
if not test_ids:
    print("test introuvable localement — telecharge-le pour verifier (recommande)")
else:
    overlap=0; seen=0
    for lang in KE_LANGS:
        for f in (train_files[lang][:2] + dev_files[lang][:1]):
            try:
                df=pd.read_parquet(f, columns=["filename"])
                ids=set(df["filename"].astype(str).str.replace(".wav","",regex=False)); seen+=len(ids)
                overlap+=len(ids & test_ids)
            except Exception as e: print("skip", f.split("/")[-1], str(e)[:50])
    print(f"chevauchement TEST: {overlap}/{seen} clips echantillonnes")
    assert overlap==0, "FUITE DETECTEE — ne pas entrainer, signaler aux organisateurs"
    print("OK : aucun chevauchement detecte sur l'echantillon")

ids test charges: 41733
chevauchement TEST: 0/21007 clips echantillonnes
OK : aucun chevauchement detecte sur l'echantillon


### 4.1 — Éval fixe par langue

In [ ]:
from datasets import load_dataset, Audio
import soundfile as sf, io, base64

# Si eval_per_lang existe deja (meme session) et qu'on veut le garder identique -> ne PAS relancer.
# Sinon on (re)construit depuis le dev officiel present sur le Drive.
def duree_bytes(b):
    try: i=sf.info(io.BytesIO(b)); return i.frames/i.samplerate
    except Exception:
        try: i=sf.info(io.BytesIO(base64.b64decode(b))); return i.frames/i.samplerate
        except Exception: return 999.0

def infos(ex):
    t=(ex.get("transcription") or "").strip()
    b=ex["audio"].get("bytes") if isinstance(ex["audio"],dict) else None
    dur=duree_bytes(b) if b else 999.0
    ex["ok"]=bool(t) and dur<=MAX_AUDIO_S
    ex["labels"]=tokenizer(clean_text(t)).input_ids if t else []
    return ex

eval_per_lang={}
for lang in KE_LANGS:
    dv = dev_files.get(lang, [])
    if not dv:
        print(f"eval {lang}: 0 dev trouve (structure sans /dev/ ?) — on prendra le 1er parquet train en secours")
        dv = train_files[lang][:1]
    picks = [p for p in dv if "unscripted" not in os.path.basename(p)][:1] + \
            [p for p in dv if "unscripted" in os.path.basename(p)][:1]
    if not picks: picks = dv[:2]
    d=load_dataset("parquet", data_files=picks, split="train")
    d=d.cast_column("audio", Audio(decode=False))
    d=d.map(infos).filter(lambda ok: ok, input_columns=["ok"])
    d=d.remove_columns([c for c in d.column_names if c not in ("audio","labels")])
    d=d.shuffle(seed=42).select(range(min(60,len(d))))
    eval_per_lang[lang]=d
    print(f"eval {lang} (dev): {len(d)}")

### 4.2 — Swahili (rechargé du Drive s'il existe, tranche du pool)

In [8]:
import json, tarfile, subprocess, os
from huggingface_hub import hf_hub_download
from datasets import Dataset, load_from_disk

SAVE_SWA="/content/drive/MyDrive/afrivoices/prepared_v5/swa"
if os.path.isdir(SAVE_SWA):
    swa_ds=load_from_disk(SAVE_SWA); print("swahili rechargé:", len(swa_ds))
else:
    SWA_REPO="DigitalUmuganda/Afrivoice_Swahili"; SWA_TEXT=["normalized_transcription","transcription"]
    TMP="/content/swa_tmp"; os.makedirs(TMP, exist_ok=True)
    def webm_to_wav(s,dst): subprocess.run(["ffmpeg","-y","-i",s,"-ar","16000","-ac","1",dst,"-loglevel","error"],check=True)
    rows=[]
    for dom in ("agriculture","health","education","financial","government"):
        folder=f"{dom}_swahili_train"
        for n in range(2):
            try:
                man=hf_hub_download(SWA_REPO,f"{folder}/manifest_{n}.jsonl",repo_type="dataset")
                tarp=hf_hub_download(SWA_REPO,f"{folder}/audio/audio_{n}.tar.xz",repo_type="dataset")
            except Exception as e: print("skip",folder,n,str(e)[:50]); continue
            want={}
            with open(man,encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    d=json.loads(line); txt=next((d[k] for k in SWA_TEXT if d.get(k)),None)
                    key=d.get("key") or os.path.splitext(os.path.basename(d.get("audio_filepath","")))[0]
                    if txt and key: want[key]=txt.strip()
            ex=os.path.join(TMP,folder,f"s{n}"); os.makedirs(ex,exist_ok=True); done=0
            with tarfile.open(tarp,"r:xz") as t:
                for m in t:
                    if done>=1000: break
                    if not m.isfile() or not m.name.endswith(".webm"): continue
                    key=os.path.splitext(os.path.basename(m.name))[0]
                    if key not in want: continue
                    fo=t.extractfile(m)
                    if fo is None: continue
                    webm=os.path.join(ex,key+".webm"); wav=os.path.join(ex,key+".wav")
                    with open(webm,"wb") as o: o.write(fo.read())
                    try: webm_to_wav(webm,wav); os.remove(webm)
                    except Exception: continue
                    with open(wav,"rb") as f: b=f.read()
                    rows.append({"audio":{"bytes":b,"path":None},
                                 "labels":tokenizer(clean_text(want[key])).input_ids})
                    done+=1
            print(f"{folder} s{n}: +{done}")
    swa_ds=Dataset.from_dict({"audio":[r["audio"] for r in rows],
                              "labels":[r["labels"] for r in rows]})
    os.makedirs(os.path.dirname(SAVE_SWA), exist_ok=True)
    swa_ds.save_to_disk(SAVE_SWA); print("swahili préparé+sauvegardé:", len(swa_ds))

sp=swa_ds.train_test_split(test_size=60, seed=42)
swa_pool=sp["train"]
swa_train_ds=swa_pool.filter(lambda ex,i: i % NB_TRANCHES == TRANCHE, with_indices=True)
eval_per_lang["swa"]=sp["test"].cast_column("audio", Audio(decode=False))
print("swa train (tranche):", len(swa_train_ds), "| swa eval:", len(eval_per_lang["swa"]))

swahili rechargé: 10000
swa train (tranche): 1657 | swa eval: 60


## 5. Cache local de la tranche — par paquets de 8 parquets, purge entre chaque

Empreinte transitoire des bruts ≈ 6 Go ; l'Arrow s'accumule (~125 Go pour la tranche).
**Aucun map/filter ensuite** (zéro copie) — le scan §5.1 fait le reste en RAM.

In [ ]:
from datasets import load_dataset, concatenate_datasets, Audio
import shutil

CHUNK = 8
lang_ranges = {}
parts = []; offset = 0
for lang in KE_LANGS:
    files = train_files[lang]
    if not files: continue
    lang_parts = []
    for i in range(0, len(files), CHUNK):
        d = load_dataset("parquet", data_files=files[i:i+CHUNK], split="train")
        d = d.cast_column("audio", Audio(decode=False))
        lang_parts.append(d)
        t, u, f = shutil.disk_usage("/content")
        print(f"  {lang} paquet {i//CHUNK+1}/{(len(files)+CHUNK-1)//CHUNK}: +{len(d)} | libre {f/1e9:.0f} Go")
        if f/1e9 < 15:
            raise RuntimeError("Disque < 15 Go : augmente NB_TRANCHES et recommence la tranche")
    dl = concatenate_datasets(lang_parts)
    lang_ranges[lang] = (offset, offset + len(dl)); offset += len(dl)
    parts.append(dl)
    print(f"{lang}: {len(dl)} clips")
ke_tranche = concatenate_datasets(parts)
print("\nTRANCHE", TRANCHE, ":", len(ke_tranche), "clips |", lang_ranges)

### 5.1 — Scan léger (durées + labels en RAM, ZÉRO copie Arrow)

Itère par lots, lit l'en-tête WAV des bytes et tokenise la transcription. Résultats en
RAM (quelques centaines de Mo max), le cache Arrow n'est jamais réécrit.

In [ ]:
ok_flags=[]; all_labels=[]
BS=256
for start in range(0, len(ke_tranche), BS):
    batch = ke_tranche[start:start+BS]      # dict de colonnes
    for a, t in zip(batch["audio"], batch["transcription"]):
        txt=(t or "").strip()
        b=a.get("bytes") if isinstance(a,dict) else None
        dur=duree_bytes(b) if b else 999.0
        ok=bool(txt) and dur<=MAX_AUDIO_S
        ok_flags.append(ok)
        all_labels.append(tokenizer(clean_text(txt)).input_ids if ok else [])
    if (start//BS) % 50 == 0:
        print(f"  scan {start}/{len(ke_tranche)}")
print("scan fini | gardés:", sum(ok_flags), "/", len(ok_flags))

## 6. Wrapper PyTorch + indices à température + collators

`TrainWrap` associe audio (mmap Arrow) et labels (RAM) sans copie. Les indices d'époque
sur-échantillonnent les langues rares selon la température (équivalent interleave).

In [ ]:
import torch, numpy as np
from dataclasses import dataclass
import soundfile as sf, librosa, random, io, base64

# indices valides par langue (kényan) + swahili en fin
groups={}
for lang,(a,b) in lang_ranges.items():
    groups[lang]=[i for i in range(a,b) if ok_flags[i]]
N_KE=len(ke_tranche)
groups["swa"]=list(range(N_KE, N_KE+len(swa_train_ds)))   # swahili indexé après

def build_epoch_indices(groups, probs, seed):
    rng=np.random.default_rng(seed)
    present={l:v for l,v in groups.items() if v}
    tot=sum(len(v) for v in present.values())
    pr={l:probs[l] for l in present}; s=sum(pr.values()); pr={l:p/s for l,p in pr.items()}
    out=[]
    for l,idxs in present.items():
        target=max(1,int(tot*pr[l]))
        reps=int(np.ceil(target/len(idxs)))
        arr=np.tile(np.array(idxs), reps)[:target]
        out.append(arr)
        print(f"  {l}: {len(idxs)} uniques -> {target} tirages (x{target/len(idxs):.2f})")
    allidx=np.concatenate(out); rng.shuffle(allidx)
    return allidx.tolist()

epoch_indices=build_epoch_indices(groups, probs, SEED)
print("époque tranche:", len(epoch_indices), "échantillons")

class TrainWrap(torch.utils.data.Dataset):
    def __init__(self, ke, swa, labels, indices, n_ke):
        self.ke=ke; self.swa=swa; self.labels=labels; self.idx=indices; self.n_ke=n_ke
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j=self.idx[i]
        if j < self.n_ke:
            return {"audio": self.ke[j]["audio"], "labels": self.labels[j]}
        k=j-self.n_ke
        ex=self.swa[k]
        return {"audio": ex["audio"], "labels": ex["labels"]}

train_ds=TrainWrap(ke_tranche, swa_train_ds, all_labels, epoch_indices, N_KE)

def decode_bytes(a):
    b=a.get("bytes")
    if not b: raise ValueError("vide")
    try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
    except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
    if arr.ndim>1: arr=arr.mean(axis=1)
    if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
    return arr.astype(np.float32)

def spec_augment_doux(feats, F=8, T=10):
    feats=feats.copy(); Tlen,D=feats.shape
    f=random.randint(0,F); f0=random.randint(0,max(0,D-f)); feats[:,f0:f0+f]=0
    t=random.randint(0,T); t0=random.randint(0,max(0,Tlen-t)); feats[t0:t0+t,:]=0
    return feats

@dataclass
class Collator:
    processor: object
    training: bool=True
    def __call__(self, features):
        arrays=[decode_bytes(f["audio"]) for f in features]
        b=self.processor.feature_extractor(arrays,sampling_rate=16000,return_tensors="pt",padding=True)
        if self.training:
            x=b["input_features"].numpy()
            for i in range(len(x)): x[i]=spec_augment_doux(x[i])
            b["input_features"]=torch.tensor(x)
        lb=self.processor.tokenizer.pad([{"input_ids":f["labels"]} for f in features],
                                        padding=True, return_tensors="pt")
        b["labels"]=lb["input_ids"].masked_fill(lb.attention_mask.ne(1),-100)
        return b

collator_train=Collator(processor, training=True)
collator_eval =Collator(processor, training=False)
print("wrapper + collators prêts")

## 7. Fine-tuning de la tranche (1 époque, lecture locale mmap = rapide)

Garde-fou : WER à 1500 pas ≤ celui du modèle de départ, sinon stop.

In [ ]:
import torch, gc, os
gc.collect(); torch.cuda.empty_cache()
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer
from datasets import concatenate_datasets
import jiwer

model=Wav2Vec2BertForCTC.from_pretrained(START_DIR, ctc_zero_infinity=True)
model.config.use_cache=False
print(f"Params {sum(p.numel() for p in model.parameters())/1e6:.0f}M — départ {START_DIR.split('/')[-1]}")

def preprocess_logits_for_metrics(logits, labels): return logits.argmax(dim=-1)
def compute_metrics(pred):
    ids=pred.predictions; labels=pred.label_ids
    labels[labels==-100]=processor.tokenizer.pad_token_id
    return {"wer": jiwer.wer(processor.batch_decode(labels, group_tokens=False),
                             processor.batch_decode(ids))}

class CleanEvalTrainer(Trainer):
    def get_eval_dataloader(self, eval_dataset=None):
        self.data_collator = collator_eval
        dl = super().get_eval_dataloader(eval_dataset)
        self.data_collator = collator_train
        return dl

eval_all=concatenate_datasets(list(eval_per_lang.values()))
use_bf16=torch.cuda.is_bf16_supported()

args=TrainingArguments(
    output_dir=OUT_DIR, remove_unused_columns=False,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    per_device_eval_batch_size=4, eval_accumulation_steps=4,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant":False},
    learning_rate=1e-5, warmup_ratio=0.03, num_train_epochs=1,
    bf16=use_bf16, fp16=(not use_bf16),
    eval_strategy="steps", eval_steps=1500,
    save_strategy="steps", save_steps=1500, save_total_limit=2,
    logging_steps=100, load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False,
    dataloader_num_workers=8, report_to="none", seed=SEED)

trainer=CleanEvalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_all,
                data_collator=collator_train, compute_metrics=compute_metrics,
                preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                processing_class=processor)

from transformers.trainer_utils import get_last_checkpoint
last=get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
if last: print("REPRISE depuis", last)
trainer.train(resume_from_checkpoint=last)
trainer.save_model(OUT_DIR); processor.save_pretrained(OUT_DIR)
print("sauvegardé ->", OUT_DIR)
nxt = TRANCHE+1
print(f"\n➡️ Prochaine session : TRANCHE = {nxt} (départ auto depuis ce modèle)"
      if nxt < NB_TRANCHES else "\n✅ TOTALITÉ parcourue. Inférence + KenLM ensuite.")

## 8. Éval propre (départ vs sortie de la tranche)

In [ ]:
import torch, jiwer
from transformers import Wav2Vec2BertForCTC

def eval_propre(model_dir, eval_per_lang, processor, max_par_langue=60):
    dev="cuda" if torch.cuda.is_available() else "cpu"
    m=Wav2Vec2BertForCTC.from_pretrained(model_dir).to(dev).eval()
    res={}
    for lang,d in eval_per_lang.items():
        preds,refs=[],[]; n=min(len(d),max_par_langue)
        with torch.no_grad():
            for j in range(n):
                ex=d[j]; arr=decode_bytes(ex["audio"])
                fe=processor.feature_extractor([arr],sampling_rate=16000,return_tensors="pt",padding=True)
                fe={k:v.to(dev) for k,v in fe.items()}
                preds.append(processor.batch_decode(torch.argmax(m(**fe).logits,dim=-1))[0])
                lab=[t for t in ex["labels"] if t!=-100]
                refs.append(processor.tokenizer.decode(lab, group_tokens=False))
        res[lang]=jiwer.wer(refs,preds); print(f"  {lang}: {res[lang]:.3f} (n={n})")
    macro=sum(res.values())/len(res); print(f"  Macro: {macro:.3f}"); return res,macro

print(f"=== départ ({START_DIR.split('/')[-1]}) ==="); eval_propre(START_DIR, eval_per_lang, processor)
print(f"\n=== sortie ({OUT_DIR.split('/')[-1]}) ==="); eval_propre(OUT_DIR, eval_per_lang, processor)

## 9. Protocole v6 & critere d'arret

Chaine : v5-t3 -> v6-t0 -> v6-t1 -> ... (TRANCHE en 2). Apres chaque tranche, la 8
compare depart vs sortie **sur le dev officiel**. Arret quand le gain < ~0.005.

Premiere session (TRANCHE=0) : la 8 donne aussi la reference **v5-t3 sur le dev
officiel** — note-la, c'est le nouveau point zero (non comparable aux anciens macros).

Puis : inference v6 + KenLM v2 (alpha 0.5, beta 1.0) -> soumission vs **0.42**.
Le depot conforme (Phase 1) reste prioritaire dans le calendrier des 10 jours.

## 10. SESSION DÉCODAGE — KenLM v2 + grille alpha/beta (après l'arrêt des tranches)

Quand les tranches sont arrêtées (gain < 0.005), cette section compare **LM v1 vs LM v2**
avec les bons hyperparamètres sur l'éval fixe, pour choisir la config de soumission.

**Mode d'emploi (session GPU fraîche)** :
1. Lance **§10.0** (installe kenlm + pyctcdecode) puis **REDÉMARRE le runtime**.
2. Après redémarrage : exécute **§1 → §4.2** (avec `TRANCHE=3` pour que l'assert passe)
   — cela reconstruit `eval_per_lang`. **NE PAS lancer §5, §6, §7** (pas de données train).
3. Lance §10.1 (logits en cache — une seule passe GPU) puis §10.2 (grille).
4. §10.3 : décision et inférence finale.

In [ ]:
# §10.0 — installer PUIS redémarrer le runtime (Exécution > Redémarrer la session)
!pip install -q pyctcdecode
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
print("⚠️ REDÉMARRE le runtime maintenant, puis exécute §1 -> §4.2 avant §10.1")

In [ ]:
# §10.1 — logits de l'éval en cache (une seule passe GPU, la grille redécode ensuite)
import torch, numpy as np, io, base64, soundfile as sf, librosa
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

BASE="/content/drive/MyDrive/afrivoices"
MODEL=f"{BASE}/baobab-asr-v6-t1"          # <- modèle final retenu
dev="cuda" if torch.cuda.is_available() else "cpu"
model=Wav2Vec2BertForCTC.from_pretrained(MODEL).to(dev).eval()
processor=Wav2Vec2BertProcessor.from_pretrained(MODEL)

def decode_bytes(a):
    b=a.get("bytes")
    if not b: raise ValueError("vide")
    try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
    except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
    if arr.ndim>1: arr=arr.mean(axis=1)
    if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
    return arr.astype(np.float32)

logits_cache={}
with torch.no_grad():
    for lang,d in eval_per_lang.items():
        rows=[]
        for j in range(len(d)):
            ex=d[j]; arr=decode_bytes(ex["audio"])
            fe=processor.feature_extractor([arr],sampling_rate=16000,return_tensors="pt",padding=True)
            lg=model(**{k:v.to(dev) for k,v in fe.items()}).logits[0].cpu().numpy()
            lab=[t for t in ex["labels"] if t!=-100]
            rows.append((lg, processor.tokenizer.decode(lab, group_tokens=False)))
        logits_cache[lang]=rows
        print(lang, len(rows), "logits en cache")

In [ ]:
# §10.2 — labels pyctcdecode + grille alpha/beta : LM v2 (3x3) vs LM v1 (référence)
from pyctcdecode import build_ctcdecoder
import jiwer
from collections import Counter

raw=[t for t,_ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank=processor.tokenizer.pad_token
labels=[]; n=0
for t in raw:
    if t==blank: labels.append("")
    elif t in ("|"," "): labels.append(" ")
    elif t in {"[UNK]","<s>","</s>"}: n+=1; labels.append("\u2047"*n)
    else: labels.append(t)
assert labels.count("")==1 and labels.count(" ")==1

def unigrams(path, top=50000):
    c=Counter()
    for line in open(path, encoding="utf-8"): c.update(line.split())
    return [w for w,_ in c.most_common(top)]

def grille(LM_DIR, alphas, betas):
    res={}
    for a in alphas:
        for b in betas:
            macros=[]
            for lang,rows in logits_cache.items():
                dec=build_ctcdecoder(labels, kenlm_model_path=f"{LM_DIR}/{lang}.bin",
                                     unigrams=unigrams(f"{LM_DIR}/{lang}.txt"), alpha=a, beta=b)
                preds=[dec.decode(lg) for lg,_ in rows]
                refs=[r for _,r in rows]
                macros.append(jiwer.wer(refs,preds))
            res[(a,b)]=sum(macros)/len(macros)
            print(f"alpha={a} beta={b} -> macro {res[(a,b)]:.4f}")
    best=min(res,key=res.get); print("MEILLEUR:", best, f"{res[best]:.4f}"); return best,res

print("=== LM v2 (grille) ===")
best_v2, res_v2 = grille(f"{BASE}/lm_v2", [0.5,0.7,0.9], [0.5,1.0,1.5])
print("\n=== LM v1 (référence 0.7/1.0) ===")
_, res_v1 = grille(f"{BASE}/lm", [0.7], [1.0])
print(f"\nRésumé: v1(0.7,1.0)={res_v1[(0.7,1.0)]:.4f} | v2{best_v2}={res_v2[best_v2]:.4f}")

In [ ]:
import torch, jiwer, numpy as np
from transformers import Wav2Vec2BertForCTC

CONFIGS = {
    "v5-t3": (f"{BASE}/baobab-asr-v5-t3", 0.5, 1.0),
    "v6-t1": (f"{BASE}/baobab-asr-v6-t1", 0.7, 0.5),
}
dev = "cuda" if torch.cuda.is_available() else "cpu"
resultats = {}
for name, (mdir, a, b) in CONFIGS.items():
    m = Wav2Vec2BertForCTC.from_pretrained(mdir).to(dev).eval()
    decs = {lang: build_ctcdecoder(labels, kenlm_model_path=f"{BASE}/lm_v2/{lang}.bin",
                                   unigrams=unigrams(f"{BASE}/lm_v2/{lang}.txt"), alpha=a, beta=b)
            for lang in eval_per_lang}
    per = {}
    with torch.no_grad():
        for lang, d in eval_per_lang.items():
            preds, refs = [], []
            for j in range(len(d)):
                ex = d[j]; arr = decode_bytes(ex["audio"])
                fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt", padding=True)
                lg = m(**{k: v.to(dev) for k, v in fe.items()}).logits[0].cpu().numpy()
                preds.append(decs[lang].decode(lg))
                refs.append(processor.tokenizer.decode([t for t in ex["labels"] if t != -100], group_tokens=False))
            per[lang] = jiwer.wer(refs, preds)
    resultats[name] = per
    del m; torch.cuda.empty_cache()
    print(name, {k: round(v,3) for k,v in per.items()}, "| macro", round(sum(per.values())/len(per),4))

print("\n=== HYBRIDE (meilleur modèle par langue) ===")
hyb = {}
for lang in eval_per_lang:
    best = min(resultats, key=lambda n: resultats[n][lang])
    hyb[lang] = resultats[best][lang]
    print(f"  {lang}: {best} ({hyb[lang]:.3f})")
print(f"  Macro hybride: {sum(hyb.values())/len(hyb):.4f}  (vs v5-t3 seul: {sum(resultats['v5-t3'].values())/6:.4f})")

### 10.3 — Décision et inférence finale

- Si **v2 au meilleur réglage < v1 (0.7/1.0)** → soumets avec **lm_v2 + best_v2**.
- Sinon → garde **lm v1 + (0.7, 1.0)** (le modèle v5-t3 apporte déjà son gain).

Inférence complète : le bloc habituel (rechargement modèle+décodeurs, decode_robuste,
tri par durées + batch dynamique, assemblage 41733 lignes) en pointant `MODEL` sur
**baobab-asr-v5-t3** et `LM` sur le gagnant, alpha/beta au réglage gagnant.
Soumets et compare à **0.45538**.

Si la grille montre un plateau serré (écarts < 0.002 entre combos), le choix exact
importe peu — prends le meilleur et passe à la soumission.

In [ ]:
import torch, jiwer, copy
from transformers import Wav2Vec2BertForCTC

m1 = Wav2Vec2BertForCTC.from_pretrained(f"{BASE}/baobab-asr-v5-t3")   # référence
m2 = Wav2Vec2BertForCTC.from_pretrained(f"{BASE}/baobab-asr-v6-t1")
sd1, sd2 = m1.state_dict(), m2.state_dict()
dev = "cuda" if torch.cuda.is_available() else "cpu"

def eval_soup(model, a, b):
    model = model.to(dev).eval()
    decs = {l: build_ctcdecoder(labels, kenlm_model_path=f"{BASE}/lm_v2/{l}.bin",
                                unigrams=unigrams(f"{BASE}/lm_v2/{l}.txt"), alpha=a, beta=b)
            for l in eval_per_lang}
    per = {}
    with torch.no_grad():
        for lang, d in eval_per_lang.items():
            preds, refs = [], []
            for j in range(len(d)):
                ex = d[j]; arr = decode_bytes(ex["audio"])
                fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt", padding=True)
                lg = model(**{k: v.to(dev) for k, v in fe.items()}).logits[0].cpu().numpy()
                preds.append(decs[lang].decode(lg))
                refs.append(processor.tokenizer.decode([t for t in ex["labels"] if t != -100], group_tokens=False))
            per[lang] = jiwer.wer(refs, preds)
    return per, sum(per.values())/len(per)

best = (None, 1.0, None)
for lam in [0.3, 0.5, 0.7]:
    soup = copy.deepcopy(m1)
    soup.load_state_dict({k: (1-lam)*sd1[k].float() + lam*sd2[k].float() for k in sd1})
    for a, b in [(0.5, 1.0), (0.7, 0.5)]:          # les deux optima connus
        per, macro = eval_soup(soup, a, b)
        print(f"λ={lam} alpha={a} beta={b} -> macro {macro:.4f} | " +
              " ".join(f"{l}:{v:.2f}" for l, v in per.items()))
        if macro < best[1]: best = ((lam, a, b), macro, copy.deepcopy(soup))
    del soup; torch.cuda.empty_cache()

(lam, a, b), macro, soup_best = best
print(f"\nMEILLEURE SOUPE: λ={lam}, alpha={a}, beta={b} -> {macro:.4f}")
print(f"Références: v5-t3 seul 0.3482 | hybride 0.3408")
if macro <= 0.3482:
    out = f"{BASE}/baobab-asr-v7-soup"
    soup_best.save_pretrained(out); processor.save_pretrained(out)
    print("sauvegardé ->", out)